In [1]:
# -*- coding: utf-8 -*-
"""
Script : Génération dataset simulé Rawbank
Auteur : Manassé Bonsomi
Description : Génère un dataset simulé pour analyser les transactions bancaires et le comportement des clients.
"""

import pandas as pd
import numpy as np

# -------------------------------
# Paramètres
# -------------------------------
n_clients = 1000  # nombre de clients simulés
np.random.seed(42)  # pour reproductibilité

# -------------------------------
# Génération des données
# -------------------------------
client_id = np.arange(1, n_clients + 1)  # identifiant unique
age = np.random.randint(18, 70, n_clients)  # âge du client
sexe = np.random.choice(['M', 'F'], n_clients)  # sexe
compte_type = np.random.choice(['Courant', 'Épargne', 'Premium'], n_clients, p=[0.5, 0.4, 0.1])
solde = np.round(np.random.normal(loc=5000, scale=2000, size=n_clients), 2)  # solde moyen avec écart type
nb_transactions_mois = np.random.poisson(lam=20, size=n_clients)  # nombre de transactions par mois
montant_total_mois = np.round(nb_transactions_mois * np.random.uniform(50, 200, n_clients), 2)  # montant total
transaction_suspecte = np.random.choice([0, 1], n_clients, p=[0.95, 0.05])  # 5% de transactions suspectes
fidelite = np.random.randint(0, 15, n_clients)  # nombre d'années comme client
produit_conso = np.random.choice(['Carte', 'Crédit', 'Assurance', 'Placement'], n_clients)

# -------------------------------
# Création du DataFrame
# -------------------------------
df_rawbank = pd.DataFrame({
    'client_id': client_id,
    'age': age,
    'sexe': sexe,
    'compte_type': compte_type,
    'solde': solde,
    'nb_transactions_mois': nb_transactions_mois,
    'montant_total_mois': montant_total_mois,
    'transaction_suspecte': transaction_suspecte,
    'fidelite': fidelite,
    'produit_conso': produit_conso
})

# -------------------------------
# Sauvegarde en CSV
# -------------------------------
df_rawbank.to_csv('rawbank_clients_simule.csv', index=False)
print("Dataset simulé généré et sauvegardé : rawbank_clients_simule.csv")

# -------------------------------
# Affichage des premières lignes
# -------------------------------
print(df_rawbank.head())


Dataset simulé généré et sauvegardé : rawbank_clients_simule.csv
   client_id  age sexe compte_type    solde  nb_transactions_mois  \
0          1   56    F     Premium  6777.97                    28   
1          2   69    F     Courant  5992.13                    23   
2          3   46    M     Courant  5111.14                    23   
3          4   32    M     Épargne  5120.47                    20   
4          5   60    F     Épargne  2121.08                    23   

   montant_total_mois  transaction_suspecte  fidelite produit_conso  
0             2153.89                     0         8        Crédit  
1             3750.26                     0        13     Placement  
2             3871.38                     0         8        Crédit  
3             1521.26                     0         1        Crédit  
4             3809.45                     0         2         Carte  


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score

# Charger le dataset simulé
df = pd.read_csv('rawbank_clients_simule.csv')

# Afficher les premières lignes
print(df.head())


In [ ]:
# Informations sur le dataset
print(df.info())
print(df.describe())

# Vérification des valeurs manquantes
print(df.isnull().sum())

# Statistiques par type de compte
print(df.groupby('compte_type')['solde', 'nb_transactions_mois'].mean())


In [ ]:
# Distribution des âges
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=15, kde=True)
plt.title('Distribution des âges des clients')
plt.show()

# Solde moyen par type de compte
plt.figure(figsize=(8,5))
sns.barplot(x='compte_type', y='solde', data=df)
plt.title('Solde moyen par type de compte')
plt.show()

# Nombre de transactions vs montant total
plt.figure(figsize=(8,5))
sns.scatterplot(x='nb_transactions_mois', y='montant_total_mois', hue='transaction_suspecte', data=df)
plt.title('Transactions mensuelles vs Montant total (transactions suspectes en rouge)')
plt.show()

# Heatmap des corrélations
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Corrélations entre variables numériques')
plt.show()

In [ ]:
# Encodage des variables catégorielles
le_sexe = LabelEncoder()
df['sexe'] = le_sexe.fit_transform(df['sexe'])

le_compte = LabelEncoder()
df['compte_type'] = le_compte.fit_transform(df['compte_type'])

le_produit = LabelEncoder()
df['produit_conso'] = le_produit.fit_transform(df['produit_conso'])

# Standardisation des variables numériques
scaler = StandardScaler()
cols_to_scale = ['age', 'solde', 'nb_transactions_mois', 'montant_total_mois', 'fidelite']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

In [ ]:
# Variable cible : transaction_suspecte
X = df.drop(['client_id', 'transaction_suspecte'], axis=1)
y = df['transaction_suspecte']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Régression logistique (simple et interprétable)
model = LogisticRegression(class_weight='balanced')  # gérer classes déséquilibrées
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Suspecte'], yticklabels=['Normal', 'Suspecte'])
plt.title('Matrice de confusion')
plt.ylabel('Vrai')
plt.xlabel('Prévu')
plt.show()

In [ ]:
# Coefficients de la régression logistique
coeffs = pd.DataFrame({'Variable': X.columns, 'Coefficient': model.coef_[0]})
coeffs['Importance'] = np.abs(coeffs['Coefficient'])
coeffs = coeffs.sort_values(by='Importance', ascending=False)
print(coeffs)

# Insight principal :
# Les clients avec solde très bas et beaucoup de transactions inhabituelles sont plus à risque

In [ ]:
# Distribution des clients à risque
plt.figure(figsize=(6,4))
sns.countplot(x='transaction_suspecte', data=df)
plt.title('Nombre de transactions normales vs suspectes')
plt.show()

# Solde vs nombre de transactions par type de compte
plt.figure(figsize=(10,6))
sns.scatterplot(x='nb_transactions_mois', y='solde', hue='compte_type', size='fidelite', data=df)
plt.title('Solde vs Transactions par type de compte et fidélité')
plt.show()

In [ ]:
# Notebook complet : Projet d'analyse des transactions bancaires Rawbank
# Auteur : Manassé Bonsomi
# Objectif : Analyse, détection d'anomalies et recommandations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score

# -------------------------------
# 1️⃣ Charger le dataset simulé
# -------------------------------
df = pd.read_csv('rawbank_clients_simule.csv')
print(df.head())

# -------------------------------
# 2️⃣ Compréhension des données
# -------------------------------
print(df.info())
print(df.describe())
print(df.isnull().sum())

# -------------------------------
# 3️⃣ Visualisation exploratoire
# -------------------------------
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=15, kde=True)
plt.title('Distribution des âges des clients')
plt.show()

plt.figure(figsize=(8,5))
sns.barplot(x='compte_type', y='solde', data=df)
plt.title('Solde moyen par type de compte')
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(x='nb_transactions_mois', y='montant_total_mois', hue='transaction_suspecte', data=df)
plt.title('Transactions mensuelles vs Montant total (transactions suspectes en rouge)')
plt.show()

plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Corrélations entre variables numériques')
plt.show()

# -------------------------------
# 4️⃣ Nettoyage et préparation
# -------------------------------
le_sexe = LabelEncoder()
df['sexe'] = le_sexe.fit_transform(df['sexe'])

le_compte = LabelEncoder()
df['compte_type'] = le_compte.fit_transform(df['compte_type'])

le_produit = LabelEncoder()
df['produit_conso'] = le_produit.fit_transform(df['produit_conso'])

scaler = StandardScaler()
cols_to_scale = ['age', 'solde', 'nb_transactions_mois', 'montant_total_mois', 'fidelite']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# -------------------------------
# 5️⃣ Séparation Train/Test
# -------------------------------
X = df.drop(['client_id', 'transaction_suspecte'], axis=1)
y = df['transaction_suspecte']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# -------------------------------
# 6️⃣ Modélisation : Régression Logistique
# -------------------------------
model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Suspecte'], yticklabels=['Normal', 'Suspecte'])
plt.title('Matrice de confusion')
plt.ylabel('Vrai')
plt.xlabel('Prévu')
plt.show()

# -------------------------------
# 7️⃣ Interprétation
# -------------------------------
coeffs = pd.DataFrame({'Variable': X.columns, 'Coefficient': model.coef_[0]})
coeffs['Importance'] = np.abs(coeffs['Coefficient'])
coeffs = coeffs.sort_values(by='Importance', ascending=False)
print(coeffs)

# -------------------------------
# 8️⃣ Visualisation finale / Dashboard simplifié
# -------------------------------
plt.figure(figsize=(6,4))
sns.countplot(x='transaction_suspecte', data=df)
plt.title('Nombre de transactions normales vs suspectes')
plt.show()

plt.figure(figsize=(10,6))
sns.scatterplot(x='nb_transactions_mois', y='solde', hue='compte_type', size='fidelite', data=df)
plt.title('Solde vs Transactions par type de compte et fidélité')
plt.show()

print("Projet complet exécuté avec succès !")